# 06주 · 수치미분·스텝·자료의 오차

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

**수업**: 2027-04-06 수치미분 (1) · 대면 / 2027-04-08 수치미분 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week06/CE273_W06.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week06)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def difference(f, x, h, method="central"):
    if not np.isfinite([x,h]).all() or h<=0:
        raise ValueError("유한한 x·양의 h 필요")
    if method=="forward":return (f(x+h)-f(x))/h
    if method=="backward":return (f(x)-f(x-h))/h
    if method=="central":return (f(x+h)-f(x-h))/(2*h)
    if method=="central4":return (-f(x+2*h)+8*f(x+h)-8*f(x-h)+f(x-2*h))/(12*h)
    raise ValueError("지원하는 차분 방법 확인 필요")


## S01. 원본 다항식의 차분 공식

**슬라이드 연결**: 2026_S1_NA_Wk6a_Differentiation1.pptx p5, 7, 9, 12, 19, 20, 21

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 차분의 방향·차수·필요한 자료점 수는?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
f=lambda x:-.1*x**4-.15*x**3-.5*x**2-.25*x+1.2
exact=-.9125
for method in ["forward","backward","central","central4"]:
    result=difference(f,.5,.25,method)
    print(method,result,"error:",abs(result-exact))
assert np.isclose(difference(f,.5,.25,"central"),-.934375)
assert np.isclose(difference(f,.5,.25,"central4"),exact)

**조건 변경**: 같은 h에서 저차·고차 공식의 필요 범위와 오차 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 원본 f·x=.5·h=.25·중앙차분/고차공식·해석 도함수·단위 명시

**결과 검토**: 경계에서 중앙차분의 자료점이 존재하는지 먼저 확인

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. 원본 sin·(x−1)^4와 h 실험

**슬라이드 연결**: 2026_S1_NA_Wk6b_Differentiation2.pptx p3, 5, 10, 16

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: h를 무조건 작게 하면 더 정확한가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
x=.5;hs=np.logspace(-1,-12,12)
errors=[abs(difference(np.sin,x,h)-np.cos(x)) for h in hs]
print(pd.DataFrame({"h":hs,"absolute_error":errors}))
f=lambda x:x**4-4*x**3+6*x**2-4*x+1
print("원본 두 번째 다항식:",difference(f,.5,.01),"analytic:",4*(.5-1)**3)
plt.loglog(hs,errors,"o-");plt.xlabel("h");plt.ylabel("absolute derivative error");plt.show()
assert min(errors)<errors[0] and np.isclose(4*(.5-1)**3,-.5)

**조건 변경**: 함수값에 작은 관측잡음을 추가한 사본에서 미분 오차 관찰

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 차분·보간·해석 미분·라이브러리 도함수의 방법 차이를 명시

**결과 검토**: 잡음·상쇄·반올림 때문에 더 작은 h가 더 나쁠 수 있음

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

차분 공식·수렴 차수·잡음·반올림의 관계 기록

입력 항목:

- `원본_참도함수`: 해당 계산값·항 수·오차 기준의 수치
- `중앙차분_h0_25`: 해당 계산값·항 수·오차 기준의 수치
- `h가_작을수록_항상_정확`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'원본_참도함수': None, '중앙차분_h0_25': None, 'h가_작을수록_항상_정확': None}


In [ ]:
def check_answers(values):
    expected = {'원본_참도함수': -0.9125, '중앙차분_h0_25': -0.934375, 'h가_작을수록_항상_정확': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
